# 🔮 Lab 5: Digital Twin Synchronization, Residuals & Diagnostics
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/blaze505050/drone-digital-twin/blob/main/examples/labs/Lab5_Digital_Twin_Sync_and_Diagnostics.ipynb)

Welcome to **Lab 5 of the DronePy Aerospace & Robotics Curriculum**!
In this final lab, you will explore the cutting edge of industrial cyber-physical systems: the **Closed-Loop Digital Twin**. You will run parallel shadow physics simulations, track state residuals $\mathbf{r}(t)$, isolate actuator failures, and understand **bounded online system identification** with condition-number gating.

---
### 🎯 Learning Objectives
1. Implement the digital twin shadow prediction paradigm alongside physical telemetry.
2. Formulate tracking residual vectors $\mathbf{r}_p(t)$ and $\mathbf{r}_v(t)$ for anomaly detection.
3. Distinguish between environmental disturbances (wind gusts) and mass/drag parameter shifts.
4. Understand DronePy's `DigitalTwinCalibrator`, condition number gating ($\kappa < 100$), and the **Persistent Excitation (PE)** condition in aerospace system ID.


In [ ]:
# Setup dependencies
try:
    import dronepy
except ImportError:
    !pip install -q git+https://github.com/blaze505050/drone-digital-twin.git
    import dronepy

import numpy as np
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

print(f"DronePy Version: {dronepy.__version__}")


---
## 1. Mathematical Theory: Digital Twin Residuals & Calibration

### A. The Shadow Digital Twin Architecture
The digital twin maintains a continuous mathematical model $\dot{\hat{\mathbf{x}}} = f(\hat{\mathbf{x}}, \mathbf{u}, \hat{\mathbf{\theta}})$ executed synchronously with telemetry from the physical UAV $\mathbf{x}_{\text{real}}(t)$.

### B. Tracking Residuals
At each telemetry packet $k$:
$$\mathbf{r}_p(t_k) = \mathbf{p}_{\text{real}}(t_k) - \hat{\mathbf{p}}_{\text{twin}}(t_k)$$
$$\mathbf{r}_v(t_k) = \mathbf{v}_{\text{real}}(t_k) - \hat{\mathbf{v}}_{\text{twin}}(t_k)$$

The Root-Mean-Square Error (RMSE) quantifies twin health:
$$\text{RMSE}_p = \sqrt{\frac{1}{N} \sum_{k=1}^N \|\mathbf{r}_p(t_k)\|^2}$$

### C. Persistent Excitation & Bounded System Identification
When parameter drift occurs, the calibrator estimates parameter updates $\Delta \mathbf{\theta}$ from the regressor matrix $\mathbf{\Phi}$:
$$\mathbf{y} = \mathbf{\Phi} \mathbf{\theta}$$
If the vehicle is in steady hover, the regressor matrix is rank-deficient and ill-conditioned ($\kappa(\mathbf{\Phi}^T \mathbf{\Phi}) \gg 100$). DronePy's safety gateway checks the condition number:
$$\kappa < 100$$
If ill-conditioned, parameter updates are **rejected**, preventing unphysical parameter divergence!


In [ ]:
# 2. Comparing a Nominal Twin to a Degraded Physical Vehicle
# Let d_twin be the nominal digital twin (1.50 kg)
# Let d_real be the physical drone which has an unmodeled 0.08 kg extra payload (1.58 kg)
d_twin = dronepy.Drone.quadcopter(mass=1.50)
d_real = dronepy.Drone.quadcopter(mass=1.58)

f_twin = d_twin.simulate(duration=2.0)
f_real = d_real.simulate(duration=2.0)

# Compute twin comparison metrics
comp = dronepy.TwinComparison.compare(f_real, f_twin)

print(f"Position RMSE:           {comp.rmse_position:.4f} m")
print(f"Velocity RMSE:           {comp.rmse_velocity:.4f} m/s")
print(f"Max Position Divergence: {comp.max_position_error:.4f} m")


---
## 3. Bounded Online Calibration & Safety Gating
We now deploy DronePy's `DigitalTwinCalibrator` and observe the safety gating mechanism.


In [ ]:
calibrator = dronepy.DigitalTwinCalibrator()
calibrated_drone, cal_result = calibrator.calibrate(d_twin, f_real)

print(cal_result.summary())
print(f"Original Twin Mass:   {cal_result.nominal_params['mass_kg']:.3f} kg")
print(f"Gate Condition Status: {'ACCEPTED' if cal_result.success else 'REJECTED (Ill-conditioned regressor protected vehicle)'}")


---
## 📝 Student Exercise: Diagnosing Vehicle Divergence

### Scenario:
A physical drone exhibits persistent position tracking residuals. You must diagnose whether the vehicle has diverged from the digital twin model and verify that safety gates prevent unphysical updates during unexcited flight.

### Your Tasks:
1. Run a 2.0s simulation with a nominal twin and a physical drone with a known mass offset.
2. Verify that `TwinComparison.compare` accurately captures the non-zero position residual.
3. Apply `DigitalTwinCalibrator` and verify that the nominal parameters match the vehicle design ($1.50\text{ kg}$).


In [ ]:
# ══════════════════════════════════════════════════════════════════
# STUDENT SOLUTION CELL - Execute digital twin diagnostics:
# ══════════════════════════════════════════════════════════════════
twin_model = dronepy.Drone.quadcopter(mass=1.50)
real_drone = dronepy.Drone.quadcopter(mass=1.62) # Physical vehicle is heavier

f_t = twin_model.simulate(duration=2.0)
f_r = real_drone.simulate(duration=2.0)

comp_study = dronepy.TwinComparison.compare(f_r, f_t)

calibrator_lab = dronepy.DigitalTwinCalibrator()
cal_drone, cal_res = calibrator_lab.calibrate(twin_model, f_r)

print(f"Observed Residual RMSE: {comp_study.rmse_position:.4f} m")
print(f"Nominal Mass:           {cal_res.nominal_params['mass_kg']:.2f} kg")

# Verification Assertions
assert comp_study.rmse_position > 0.0, "Mass mismatch must cause non-zero position residual"
assert cal_res.nominal_params["mass_kg"] == 1.50, "Nominal mass must be 1.50 kg"
assert len(comp_study.position_residuals) == len(comp_study.time)
print("SUCCESS: Lab 5 Digital Twin diagnostics verified successfully!")
